In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Add the root directory to Python's path if it isn't there already
if PROJECT_ROOT not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from autoosu_reference.models import OsuModelNpPred, OsuModel
from autoosu_reference.loss import binary_focal_loss, multi_focal_loss
from tracers.datasets import (
    AutoOsu4KDataset,
    split_song_ids,
    make_dataloader,
)



device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060


In [10]:
ROOT = PROJECT_ROOT / "dataset"

AUDIO_DIR = ROOT / "audio"
BEATMAP_DIR = ROOT / "beatmap" / "4keys"

In [11]:
base_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=1000,
    crop_mode="random",
    audio_cache_size=8,
)

print("Charts:", len(base_dataset))
print("Unique songs:", len(base_dataset.song_ids))

Charts: 1126
Unique songs: 375


In [12]:
train_ids, val_ids, test_ids = split_song_ids(
    base_dataset.song_ids,
    val_fraction=0.10,
    test_fraction=0.10,
    seed=42,
)

print("Training songs:", len(train_ids))
print("Validation songs:", len(val_ids))
print("Test songs:", len(test_ids))

Training songs: 299
Validation songs: 38
Test songs: 38


In [13]:
train_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=1000,
    crop_mode="random",
    crops_per_chart=2,
    song_ids=train_ids,
)

val_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=1000,
    crop_mode="all",
    song_ids=val_ids,
)

test_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=1000,
    crop_mode="all",
    song_ids=test_ids,
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Training samples: 1796
Validation samples: 1760
Test samples: 1668


In [15]:
train_loader = make_dataloader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
)

val_loader = make_dataloader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

test_loader = make_dataloader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

In [14]:
hyperparams = {
    "bp_emb_dim": 32,
    "bn_emb_dim": 16,
    "diff_emb_dim": 16,

    "np_hidden_size": 256,
    "np_num_layers": 2,

    "ns_pre_proj_size": 32,
    "ns_hidden_size": 256,
    "ns_num_layers": 2,

    "action_emb_dim": 32,
}

model = OsuModelNpPred(hyperparams).to(device)

print(model)

OsuModelNpPred(
  (gelu): GELU(approximate='none')
  (sigmoid): Sigmoid()
  (stack): Sequential(
    (0): Conv2d(3, 8, kernel_size=(5, 3), stride=(1, 2), padding=(2, 1))
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(8, 16, kernel_size=(5, 3), stride=(1, 2), padding=(2, 1))
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(16, 32, kernel_size=(5, 3), stride=(1, 2), padding=(2, 1))
    (7): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): Conv2d(32, 64, kernel_size=(5, 3), stride=(1, 2), padding=(2, 1))
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (11): ReLU()
  )
  (beat_phase_emb): Embedding(49, 32)
  (beat_num_emb): Embedding(4, 16)
  (difficulty_proj): Sequential(
    (0): Linear(in_features=1, 

In [16]:
batch = next(iter(train_loader))

specs = batch["specs"].to(device)
beat_phases = batch["beat_phase"].to(device)
beat_nums = batch["beat_num"].to(device)
difficulties = batch["difficulty"].to(device)

actions_input = batch["actions_input"].to(device)

onsets_target = batch["onsets"].to(device)
actions_target = batch["actions_target"].to(device)